# SubVoice GPU server (free Colab / Kaggle GPU)
Run the cells **in order**, keep this tab open, and paste the printed `https://….trycloudflare.com` link into the SubVoice app (*Chatterbox server URL*).
Runtime -> Change runtime type -> **GPU** (T4). Free sessions are time-limited and the link changes every session; the app re-uploads your cloned voice automatically.

In [ ]:
!nvidia-smi | head -n 15
!apt-get -qq install -y ffmpeg > /dev/null && echo ffmpeg-ok

In [ ]:
!pip -q install chatterbox-tts fastapi uvicorn soundfile python-multipart

In [ ]:
%%writefile /content/app.py
"""
SubVoice TTS server.
  ENGINES=xtts       -> serves POST /tts/xtts         (Coqui XTTS-v2, built-in male/female voices)
  ENGINES=chatterbox -> serves POST /tts/chatterbox   (Chatterbox Multilingual, Hindi voice cloning)
  ENGINES=xtts,chatterbox -> both (only if their pip dependencies coexist; otherwise run two venvs)
Always available: POST /voice (upload reference clip), GET /health
"""
import io, os, subprocess, tempfile, threading
import numpy as np
import soundfile as sf
import torch
from fastapi import FastAPI, File, Form, HTTPException, UploadFile
from fastapi.responses import Response
from pydantic import BaseModel

ENGINES = os.getenv("ENGINES", "xtts,chatterbox").split(",")
DEV = "cuda" if torch.cuda.is_available() else "cpu"
VOICES = os.path.abspath("voices"); os.makedirs(VOICES, exist_ok=True)
XTTS_SPEAKERS = {"female": "Claribel Dervla", "male": "Damien Black"}

app = FastAPI()
lock = threading.Lock()
_xtts = None
_cb = None


class Req(BaseModel):
    text: str
    language: str = "hi"
    gender: str = "female"
    voice_id: str = ""


def voice_path(vid: str):
    if not vid:
        return None
    p = os.path.join(VOICES, os.path.basename(vid) + ".wav")
    return p if os.path.exists(p) else None


def wav_response(wav, sr):
    if hasattr(wav, "detach"):
        wav = wav.detach().cpu().numpy()
    arr = np.asarray(wav, dtype="float32").squeeze()
    buf = io.BytesIO()
    sf.write(buf, arr, sr, format="WAV")
    return Response(buf.getvalue(), media_type="audio/wav")


@app.get("/health")
def health():
    return {"engines": ENGINES, "device": DEV}


@app.post("/voice")
async def upload_voice(voice_id: str = Form(...), file: UploadFile = File(...)):
    raw = await file.read()
    with tempfile.NamedTemporaryFile(delete=False) as t:
        t.write(raw)
    out = os.path.join(VOICES, os.path.basename(voice_id) + ".wav")
    # normalise to mono 24 kHz, max 20 s (needs ffmpeg installed)
    r = subprocess.run(["ffmpeg", "-y", "-i", t.name, "-t", "20", "-ar", "24000", "-ac", "1", out],
                       capture_output=True)
    os.unlink(t.name)
    if r.returncode != 0:
        raise HTTPException(400, "could not decode audio: " + r.stderr.decode()[-200:])
    return {"voice_id": voice_id}


def load_xtts():
    global _xtts
    if _xtts is None:
        from TTS.api import TTS
        _xtts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(DEV)


def load_cb():
    global _cb
    if _cb is None:
        from chatterbox.mtl_tts import ChatterboxMultilingualTTS
        _cb = ChatterboxMultilingualTTS.from_pretrained(device=DEV)


@app.on_event("startup")
def warm_up():
    if "xtts" in ENGINES: load_xtts()
    if "chatterbox" in ENGINES: load_cb()


@app.post("/tts/xtts")
def tts_xtts(r: Req):
    if "xtts" not in ENGINES:
        raise HTTPException(404, "xtts not enabled on this server")
    with lock:
        load_xtts()
        speaker = XTTS_SPEAKERS.get(r.gender, XTTS_SPEAKERS["female"])
        wav = _xtts.tts(text=r.text, language=r.language, speaker=speaker)
    return wav_response(wav, 24000)


@app.post("/tts/chatterbox")
def tts_chatterbox(r: Req):
    if "chatterbox" not in ENGINES:
        raise HTTPException(404, "chatterbox not enabled on this server")
    with lock:
        load_cb()
        wav = _cb.generate(r.text, language_id="hi",            # Hindi only
                           audio_prompt_path=voice_path(r.voice_id))
        sr = _cb.sr
    return wav_response(wav, sr)

In [ ]:
import subprocess, os, re, time
os.makedirs("/content/work", exist_ok=True); os.chdir("/content/work")
env = dict(os.environ, ENGINES="chatterbox")
srv = subprocess.Popen(["python","-m","uvicorn","app:app","--app-dir","/content","--host","0.0.0.0","--port","8001"],
                       env=env, stdout=open("srv.log","w"), stderr=subprocess.STDOUT)
# wait for model download + warm-up
import urllib.request
for i in range(180):
    try:
        print(urllib.request.urlopen("http://127.0.0.1:8001/health").read()); break
    except Exception: time.sleep(5)
else: print(open("srv.log").read()[-3000:])

In [ ]:
!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared
import subprocess, re, time
t = subprocess.Popen(["/content/cloudflared","tunnel","--url","http://127.0.0.1:8001"], stderr=subprocess.PIPE, text=True)
url=None
while url is None:
    line=t.stderr.readline()
    m=re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m: url=m.group(0)
print("\n>>> Paste this into the SubVoice app (Chatterbox server URL):\n", url)

## Optional: XTTS-v2 server (separate environment, port 8000)
Only needed if you want the `xtts` engine. The offline `device` engine already gives male/female Hindi on the tablet, so most people can skip this.

In [ ]:
!apt-get -qq install -y python3-venv > /dev/null
!python3 -m venv /content/xv
!/content/xv/bin/pip -q install coqui-tts fastapi uvicorn soundfile python-multipart
import subprocess, os
env = dict(os.environ, ENGINES="xtts", COQUI_TOS_AGREED="1")   # XTTS licence is non-commercial (CPML)
subprocess.Popen(["/content/xv/bin/python","-m","uvicorn","app:app","--app-dir","/content","--host","0.0.0.0","--port","8000"], env=env,
                 stdout=open("/content/work/xtts.log","w"), stderr=subprocess.STDOUT)

In [ ]:
import subprocess, re
t2 = subprocess.Popen(["/content/cloudflared","tunnel","--url","http://127.0.0.1:8000"], stderr=subprocess.PIPE, text=True)
while True:
    m=re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", t2.stderr.readline())
    if m: print(">>> XTTS URL:", m.group(0)); break